# IMDb Sentiment Analysis — AWS SageMaker

**Pipeline:**
1. Load dataset from S3
2. ETL — text preprocessing
3. EDA — exploratory analysis
4. Feature Engineering — TF-IDF
5. Model Training — Logistic Regression & Random Forest with GridSearch
6. Model Evaluation — metrics, confusion matrix, ROC curve
7. Save model artifacts to S3 for Lambda deployment

## 0. Setup

In [ ]:
!pip install -q wordcloud nltk scikit-learn pandas numpy matplotlib seaborn boto3

In [ ]:
import boto3
import pandas as pd
import numpy as np
import re
import pickle
import io
import json
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import ngrams

from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, roc_curve, auc,
    precision_recall_fscore_support
)

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print('All imports successful')

In [ ]:
# ======================================================
# CHANGE THIS TO YOUR S3 BUCKET NAME
S3_BUCKET = 'imdb-sentiment-YOUR-NAME'
# ======================================================

S3_DATA_KEY    = 'data/IMDB Dataset.csv'
S3_MODEL_PREFIX = 'models/'

s3 = boto3.client('s3')
print(f'Using bucket: s3://{S3_BUCKET}')

## 1. Data Loading from S3

In [ ]:
print(f'Loading dataset from s3://{S3_BUCKET}/{S3_DATA_KEY} ...')
obj = s3.get_object(Bucket=S3_BUCKET, Key=S3_DATA_KEY)
df = pd.read_csv(obj['Body'])

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

## 2. Exploratory Data Analysis (EDA)

In [ ]:
print('=== Dataset Info ===')
print(df.info())
print(f'\nMissing values:\n{df.isnull().sum()}')
print(f'\nSentiment distribution:\n{df["sentiment"].value_counts()}')
print(f'\nClass balance: {df["sentiment"].value_counts(normalize=True).round(3).to_dict()}')

In [ ]:
df['review_length'] = df['review'].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sentiment distribution
counts = df['sentiment'].value_counts()
axes[0].bar(counts.index, counts.values, color=['green', 'red'], alpha=0.8, edgecolor='black')
axes[0].set_title('Sentiment Distribution', fontsize=14)
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')
for i, (label, count) in enumerate(counts.items()):
    axes[0].text(i, count + 100, str(count), ha='center', fontsize=12)

# Review length histogram
for sentiment, color in [('positive', 'green'), ('negative', 'red')]:
    data = df[df['sentiment'] == sentiment]['review_length']
    axes[1].hist(data, bins=50, alpha=0.5, color=color, label=sentiment)
axes[1].set_title('Review Length Distribution', fontsize=14)
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].legend()

# Boxplot
df.boxplot(column='review_length', by='sentiment', ax=axes[2])
axes[2].set_title('Review Length by Sentiment', fontsize=14)
axes[2].set_xlabel('Sentiment')
axes[2].set_ylabel('Word Count')
plt.suptitle('')

plt.tight_layout()
plt.savefig('/tmp/eda_overview.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'Average review length: {df["review_length"].mean():.0f} words')
print(f'Median review length: {df["review_length"].median():.0f} words')

## 3. ETL — Text Preprocessing

Pipeline:
- Remove HTML tags (`<br />`)
- Remove punctuation and special characters
- Convert to lowercase
- Remove English stopwords
- Lemmatization (WordNet)

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<br\s*/?>', ' ', text)          # Remove HTML tags
    text = re.sub(r'[^a-zA-Z\s]', '', text)         # Remove non-alpha
    words = text.split()
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words and len(word) > 2
    ]
    return ' '.join(words)

print('Preprocessing text...')
df['cleaned_review'] = df['review'].apply(clean_text)
df['target'] = df['sentiment'].apply(lambda x: 1 if x == 'positive' else 0)
df['cleaned_length'] = df['cleaned_review'].apply(lambda x: len(x.split()))

print(f'Preprocessing complete.')
print(f'\nBefore cleaning (avg words): {df["review_length"].mean():.0f}')
print(f'After cleaning (avg words):  {df["cleaned_length"].mean():.0f}')
df[['review', 'cleaned_review', 'sentiment', 'target']].head(3)

In [ ]:
pos_text = ' '.join(df[df['sentiment'] == 'positive']['cleaned_review'])
neg_text = ' '.join(df[df['sentiment'] == 'negative']['cleaned_review'])

wc_pos = WordCloud(width=800, height=400, background_color='white',
                   max_words=100, colormap='Greens').generate(pos_text)
wc_neg = WordCloud(width=800, height=400, background_color='black',
                   max_words=100, colormap='Reds').generate(neg_text)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(wc_pos, interpolation='bilinear')
axes[0].set_title('Most Common Words — Positive Reviews', fontsize=14)
axes[0].axis('off')
axes[1].imshow(wc_neg, interpolation='bilinear')
axes[1].set_title('Most Common Words — Negative Reviews', fontsize=14)
axes[1].axis('off')
plt.tight_layout()
plt.savefig('/tmp/wordclouds.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
def plot_ngrams(reviews_pos, reviews_neg, n, top_k=15):
    pos_counts = Counter()
    neg_counts = Counter()
    for r in reviews_pos:
        pos_counts.update(ngrams(r.split(), n))
    for r in reviews_neg:
        neg_counts.update(ngrams(r.split(), n))

    pos_top = pos_counts.most_common(top_k)
    neg_top = neg_counts.most_common(top_k)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    name = 'Bigrams' if n == 2 else 'Trigrams'

    axes[0].barh(range(top_k), [c for _, c in pos_top][::-1], color='green', alpha=0.7)
    axes[0].set_yticks(range(top_k))
    axes[0].set_yticklabels([' '.join(ng) for ng, _ in pos_top][::-1])
    axes[0].set_title(f'Top {top_k} {name} — Positive')

    axes[1].barh(range(top_k), [c for _, c in neg_top][::-1], color='red', alpha=0.7)
    axes[1].set_yticks(range(top_k))
    axes[1].set_yticklabels([' '.join(ng) for ng, _ in neg_top][::-1])
    axes[1].set_title(f'Top {top_k} {name} — Negative')

    plt.tight_layout()
    plt.savefig(f'/tmp/{name.lower()}.png', dpi=100, bbox_inches='tight')
    plt.show()

pos_reviews = df[df['sentiment'] == 'positive']['cleaned_review']
neg_reviews = df[df['sentiment'] == 'negative']['cleaned_review']

plot_ngrams(pos_reviews, neg_reviews, n=2)
plot_ngrams(pos_reviews, neg_reviews, n=3)

## 4. Feature Engineering — TF-IDF Vectorization

In [ ]:
print('Splitting data and vectorizing with TF-IDF...')

X = df['cleaned_review']
y = df['target']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)

print(f'Train set size:   {X_train.shape}')
print(f'Test set size:    {X_test.shape}')
print(f'Vocabulary size:  {len(vectorizer.vocabulary_)} features')
print(f'Train positives:  {y_train.sum()} / {len(y_train)}')
print(f'Test positives:   {y_test.sum()} / {len(y_test)}')

## 5. Model Training

In [ ]:
print('=== Logistic Regression with GridSearch ===')

param_grid_lr = {'C': [0.01, 0.1, 1, 10], 'penalty': ['l2']}
grid_lr = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid_lr, cv=3, verbose=1, n_jobs=-1
)
grid_lr.fit(X_train, y_train)

lr_model = grid_lr.best_estimator_
print(f'Best parameters: {grid_lr.best_params_}')
print(f'Best CV score:   {grid_lr.best_score_:.4f}')

In [ ]:
print('=== Random Forest with GridSearch ===')

param_grid_rf = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}
grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid_rf, cv=3, verbose=1, n_jobs=-1
)
grid_rf.fit(X_train, y_train)

rf_model = grid_rf.best_estimator_
print(f'Best parameters: {grid_rf.best_params_}')
print(f'Best CV score:   {grid_rf.best_score_:.4f}')

## 6. Model Evaluation

In [ ]:
results = {}

for model_name, trained_model in [('Logistic Regression', lr_model), ('Random Forest', rf_model)]:
    y_pred  = trained_model.predict(X_test)
    y_proba = trained_model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')
    results[model_name] = {'Accuracy': acc, 'Precision': p, 'Recall': r, 'F1-Score': f1,
                           'y_pred': y_pred, 'y_proba': y_proba}

    print(f'\n=== {model_name} ===')
    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {p:.4f}')
    print(f'Recall:    {r:.4f}')
    print(f'F1-Score:  {f1:.4f}')
    print('\nClassification Report:')
    print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for row, (model_name, res) in enumerate(results.items()):
    y_pred  = res['y_pred']
    y_proba = res['y_proba']

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[row][0],
                xticklabels=['Negative', 'Positive'],
                yticklabels=['Negative', 'Positive'])
    axes[row][0].set_title(f'{model_name} — Confusion Matrix', fontsize=13)
    axes[row][0].set_ylabel('True Label')
    axes[row][0].set_xlabel('Predicted Label')

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    axes[row][1].plot(fpr, tpr, color='darkorange', lw=2,
                      label=f'ROC (AUC = {roc_auc:.3f})')
    axes[row][1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    axes[row][1].set_xlim([0.0, 1.0])
    axes[row][1].set_ylim([0.0, 1.05])
    axes[row][1].set_xlabel('False Positive Rate')
    axes[row][1].set_ylabel('True Positive Rate')
    axes[row][1].set_title(f'{model_name} — ROC Curve', fontsize=13)
    axes[row][1].legend(loc='lower right')

plt.tight_layout()
plt.savefig('/tmp/evaluation.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
model_names = list(results.keys())
x = np.arange(len(model_names))
width = 0.2

fig, ax = plt.subplots(figsize=(12, 6))
for i, metric in enumerate(metrics):
    values = [results[m][metric] for m in model_names]
    bars = ax.bar(x + i * width, values, width, label=metric, alpha=0.85)
    for bar in bars:
        ax.annotate(f'{bar.get_height():.3f}',
                    xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(model_names, fontsize=12)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print('\n=== Final Comparison ===')
comparison_df = pd.DataFrame({
    name: {k: v for k, v in res.items() if k in metrics}
    for name, res in results.items()
}).T.round(4)
print(comparison_df)

In [ ]:
feature_names = vectorizer.get_feature_names_out()
top_n = 20

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Logistic Regression coefficients
lr_coefs = np.abs(lr_model.coef_[0])
top_lr = np.argsort(lr_coefs)[-top_n:]
axes[0].barh(range(top_n), lr_coefs[top_lr], color='steelblue', alpha=0.8)
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels([feature_names[i] for i in top_lr])
axes[0].set_title('Top 20 Features — Logistic Regression', fontsize=13)
axes[0].set_xlabel('Absolute Coefficient')

# Random Forest feature importance
rf_importances = rf_model.feature_importances_
top_rf = np.argsort(rf_importances)[-top_n:]
axes[1].barh(range(top_n), rf_importances[top_rf], color='darkorange', alpha=0.8)
axes[1].set_yticks(range(top_n))
axes[1].set_yticklabels([feature_names[i] for i in top_rf])
axes[1].set_title('Top 20 Features — Random Forest', fontsize=13)
axes[1].set_xlabel('Feature Importance')

plt.tight_layout()
plt.savefig('/tmp/feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Save Best Model to S3 (for Lambda deployment)

In [ ]:
# Select best model by F1-Score
best_name = max(
    [(name, res['F1-Score']) for name, res in results.items()],
    key=lambda x: x[1]
)[0]
best_model = lr_model if best_name == 'Logistic Regression' else rf_model
best_params = grid_lr.best_params_ if best_name == 'Logistic Regression' else grid_rf.best_params_

print(f'Best model: {best_name}')
print(f'Accuracy:   {results[best_name]["Accuracy"]:.4f}')
print(f'F1-Score:   {results[best_name]["F1-Score"]:.4f}')

def save_to_s3(obj, bucket, key):
    buf = io.BytesIO()
    pickle.dump(obj, buf)
    buf.seek(0)
    s3.upload_fileobj(buf, bucket, key)
    print(f'Saved: s3://{bucket}/{key}')

# Save vectorizer
save_to_s3(vectorizer, S3_BUCKET, f'{S3_MODEL_PREFIX}vectorizer.pkl')

# Save model
save_to_s3(best_model, S3_BUCKET, f'{S3_MODEL_PREFIX}model.pkl')

# Save metadata
metadata = {
    'model_type': best_name,
    'best_params': best_params,
    'metrics': {k: round(v, 4) for k, v in results[best_name].items() if k in ['Accuracy','Precision','Recall','F1-Score']},
    'vocabulary_size': len(vectorizer.vocabulary_),
    'training_samples': int(X_train.shape[0]),
    'test_samples': int(X_test.shape[0]),
    'dataset_size': len(df)
}
meta_buf = io.BytesIO(json.dumps(metadata, indent=2).encode('utf-8'))
s3.upload_fileobj(meta_buf, S3_BUCKET, f'{S3_MODEL_PREFIX}metadata.json')
print(f'Saved: s3://{S3_BUCKET}/{S3_MODEL_PREFIX}metadata.json')

print('\nAll artifacts saved to S3!')
print('\nModel is ready for Lambda deployment.')
print(f'Set Lambda env var MODEL_BUCKET={S3_BUCKET}')

## 8. Quick Inference Test

In [ ]:
import re

def predict_sentiment(review_text, model, vectorizer, lemmatizer, stop_words):
    text = review_text.lower()
    text = re.sub(r'<br\s*/?>', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = [lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words and len(w) > 2]
    cleaned = ' '.join(words)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]
    proba = model.predict_proba(vec)[0]
    label = 'POSITIVE' if pred == 1 else 'NEGATIVE'
    return label, proba[pred] * 100

test_reviews = [
    "This movie was absolutely fantastic! Great performances and an amazing plot.",
    "Terrible film. Waste of time. The acting was awful and the story made no sense.",
    "It was okay, nothing special but not bad either."
]

print('=== Inference Test ===')
for review in test_reviews:
    label, confidence = predict_sentiment(review, best_model, vectorizer, lemmatizer, stop_words)
    print(f'Review: "{review[:60]}..."')
    print(f'Result: {label} (confidence: {confidence:.1f}%)')
    print()